In [ ]:
///- --package .

#### summon (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:< sm'_operators
:= orders_of (text : string) : array string =
    text |> sm'::split ";" |> array |> am'::choose \ part => ? part = "" ?> None |: Some part

:= order_count (orders : array string) : i32 = am'::length orders

:= order_at (orders : array string) (position : i32) : string = am'::index (position - 1) orders

:= creature_line (c : client::creature) : string =
    := name = client::creature_name c
    := monster = ? client::creature_is_monster c ?> "true" |: "false"
    "sees " ++# name ++# " monster=" ++# monster ++# " at " ++# probe::described (client::creature_spot c)

:~ ^ creatures_from (people : client::crowd) (size : i32) (index : i32) : () =
    ? index <= size ?>
        probe::note (creature_line (client::member people index))
        creatures_from people size (index + 1)

:= list_creatures () : () =
    := people = client::spectators (client::player_spot ())
    creatures_from people (client::crowd_size people) 1

:= order_scheduled (orders : array string) (index : i32) : () =
    := order = order_at orders index
    client::after (1200 + 500 * (index - 1)) (\ () => client::talk ("/m " ++# order))

:= orders_scheduled (orders : array string) : () =
    := count = order_count orders
    :~ ^ schedule (index : i32) : () =
        ? index <= count ?>
            order_scheduled orders index
            schedule (index + 1)
    schedule 1

:= listed_then_finish () : () =
    list_creatures ()
    probe::finish "summoned"

:= summon_orders (summon_at : string) (orders : array string) : () =
    probe::note ("at " ++# probe::described (client::player_spot ()))
    client::talk ("/pos " ++# summon_at ++# ", 7")
    orders_scheduled orders
    client::after 4000 listed_then_finish

:= summon_ready (summon_at : string) (orders : array string) : () =
    probe::note ("in game as " ++# client::player_name ())
    client::after 1500 \ () => summon_orders summon_at orders

:= server_text_noted (mode : string) (text : string) : () = probe::note ("server text: " ++# text)

:= summon_place () : string = client::env_text "TIBIA_SUMMON_AT" "1932, 1352"

:= wanted_orders () : array string = orders_of (client::env_text "TIBIA_SUMMON_ORDERS" "green frog,3,,2")

:= timed_out () : () = probe::finish "timeout"

#### orders_drop_empty_parts_like_gmatch (test)

In [1]:
///- --test --symbol
///> lua
///> c

(orders_of "green frog,3,,2", orders_of ";wolf;;troll,2;")
|> _assert_eq (;[ "green frog,3,,2" ], ;[ "wolf"; "troll,2" ])

.lua (lua)
__assert_eq { expected = ([green frog,3,,2], [wolf, troll,2]) }
.c (c)
__assert_eq { expected = ([green frog,3,,2], [wolf, troll,2]) }

In [ ]:
:= main () =
    := summon_at, orders = summon_place (), wanted_orders ()
    probe::started_as (probe::game_master ()) server_text_noted (\ () => summon_ready summon_at orders) 60000 timed_out
    0i32